# Gene deletion prevalence for hrp2/3
This notebook is designed to import and visualise data aggregated by `nomadic summarize`. The notebook can be run one cell at a time (Shift-Enter) or all together ('Run All' above).

In [1]:
from pathlib import Path
import pandas as pd
from typing import Optional
import plotly.graph_objects as go
import numpy as np
from statsmodels.stats.proportion import proportion_confint
import sys
import yaml

sys.path.append("../functions")
from gene_deletions import DeletionFinder
from workspace import Workspace

%load_ext autoreload
%autoreload 2

# Settings


In [2]:
# Add the full path to the location where your workspace is
workspace_man = "/path/to/your/workspace"
# Decide whether you want the outputs to be saved and in which format
save_results = True
save_format = "svg"

In [3]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

workspace_dir = Path(config.get("workspace_dir", workspace_man)).expanduser()
ws = Workspace(workspace_dir)
print(f"Workspace loaded: {ws.name}")

# Define where the outputs will be saved
output_dir = Path.cwd() / "results" / ws.name

if save_results:
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"All results will be saved to: {output_dir}")

Workspace loaded: ATSB_edited
All results will be saved to: /home/dan/git/resources/notebooks/gene_deletions/results/ATSB_edited


# Functions

In [4]:
# Copied from nomadic verbatim, except gene_deletions_df join changed from right to left
def gene_deletion_prevalence_by(
    gene_deletions_df: pd.DataFrame, master_df: pd.DataFrame, fields: list[str]
) -> pd.DataFrame:
    """
    Compute the prevalence of gene deletions in `gene_deletions_df`
    stratified by columns in `fields`.
    """
    gene_deletions_df = gene_deletions_df.merge(
        master_df[["sample_id", *fields]], on="sample_id", how="right"
    )

    prev_df = (
        gene_deletions_df.groupby(["gene", *fields])
        .agg(
            n_samples=pd.NamedAgg("is_deleted", len),
            n_passed=pd.NamedAgg("is_deleted", lambda x: sum(x.notnull())),
            n_deleted=pd.NamedAgg("is_deleted", lambda x: sum(x)),
        )
        .reset_index()
    )

    # Compute prevalence
    prev_df["prevalence"] = 100 * prev_df["n_deleted"] / prev_df["n_passed"]

    # Compute prevalence 95% confidence intervals
    low, high = proportion_confint(
        prev_df["n_deleted"],
        prev_df["n_passed"],
        alpha=0.05,
        method="beta",
    )
    prev_df["prevalence_lowci"] = 100 * low
    prev_df["prevalence_highci"] = 100 * high

    return prev_df

In [5]:
def generate_deletion_prevalence_barchart(
    deletions_df: pd.DataFrame,
    master_df: pd.DataFrame,
    by: Optional[str] = "All",
    fig_prefix: str = None,
    min_count: int = None,
) -> go.Figure:
    """
    Build a barchart from the df provided
    """
    if min_count is not None and by == "All":
        raise ValueError("min_count can only be used with a grouping variable")

    if by == "All":
        plot_df = gene_deletion_prevalence_by(deletions_df, master_df, [])
    else:
        plot_df = gene_deletion_prevalence_by(deletions_df, master_df, [by])
    
    if min_count is not None:
        plot_df = plot_df[plot_df["n_passed"] >= min_count]

    genes = set(plot_df["gene"])

    fig_name = f"Prevalence of {', '.join(genes)} mutations"

    if fig_prefix is not None:
        fig_name = f"{fig_prefix}: {fig_name}"


    data = []
    htemp = "%{y:0.1f}% (%{customdata[2]}/%{customdata[1]})"
    
    if by == "All":
        # Prepare plotting data
        customdata = np.stack(
            [
                plot_df["n_samples"],
                plot_df["n_passed"],
                plot_df["n_deleted"],
            ],
            axis=-1,
        )
        data.append(
            go.Bar(
                x=plot_df["gene"],
                y=plot_df["prevalence"],
                customdata=customdata,
                hovertemplate=htemp,
                name="Prevalence",
                error_y=dict(
                    type="data",
                    array=plot_df["prevalence_highci"] - plot_df["prevalence"],
                    arrayminus=plot_df["prevalence"]
                    - plot_df["prevalence_lowci"],
                ),
            )
        )
    else:
        for group in plot_df[by].unique():
            group_df = plot_df.query(f"{by} == @group")
            # Prepare plotting data
            customdata = np.stack(
                [
                    group_df["n_samples"],
                    group_df["n_passed"],
                    group_df["n_deleted"],
                ],
                axis=-1,
            )
            data.append(
                go.Bar(
                    x=group_df["gene"],
                    y=group_df["prevalence"],
                    customdata=customdata,
                    hovertemplate=htemp,
                    name=str(group),
                    error_y=dict(
                        type="data",
                        array=group_df["prevalence_highci"] - group_df["prevalence"],
                        arrayminus=group_df["prevalence"]
                        - group_df["prevalence_lowci"],
                    ),
                )
            )
        fig_name = (f"{fig_name} by {by.capitalize()}")

        if min_count is not None:
            fig_name += f" (min n={min_count})"

    # Plotting
    fig = go.Figure(data)
    fig.update_layout(
        yaxis_title="Prevalence (%)",
        xaxis=dict(showline=True, linewidth=1, linecolor="black", mirror=True),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            showgrid=True,
            gridcolor="lightgray",
            gridwidth=0.5,
            griddash="dot",
        ),
        legend=dict(
            orientation="h", yanchor="top", y=-0.15, xanchor="left", x=0
        ),
        plot_bgcolor="rgba(0,0,0,0)",
        hovermode="x unified",
    )
    fig.update_yaxes(range=[0, 100])
    fig.update_traces(marker=dict(line=dict(color="black", width=1)))
    fig.update_layout(title=fig_name)
    
    if save_results:
        fig.write_image(
            output_dir / f"{fig_name.replace(' ', '_')}.{save_format}",
        )
    return fig

# Prepare data

### Load and filter

In [6]:
master_df = pd.read_csv(ws.master_csv_path)
summary_cov = pd.read_csv(ws.summaries_path / "summary.coverage.csv")

In [ ]:
dfs = []
hrp_genes = ["hrp2-p14-306", "hrp3-p14-276"] 

for res_dir in ws.results_path.iterdir():
    print(f"Processing {res_dir.name}")
    cov_df = pd.read_csv(res_dir / "summary.region_coverage.csv")

    # Remove entries that have failed QC
    passed_df = summary_cov[["barcode", "name"]][(summary_cov["expt_name"] == res_dir.name) & (summary_cov["status"].str.contains("pass|control", na=False))]
    if passed_df.empty:
        print(f"WARNING: No samples passed QC for {res_dir.name}. Skipping...")
        continue
    
    cov_df_filtered = cov_df.merge(passed_df, on=["barcode", "name"], how="inner").copy(deep=True)

    del_cls = DeletionFinder(cov_df_filtered)
    exp_meta = pd.read_csv(res_dir / "metadata" / "samples.csv")
    meta_cols = ["sample_id","barcode","sample_type"]
    exp_meta = exp_meta[meta_cols]
    if "sample_type" not in exp_meta.columns:
        print(f"WARNING: No sample_type column identified. Skipping {res_dir.name}...")
        continue

    neg_bcs = list(exp_meta["barcode"][exp_meta["sample_type"]=="neg"])
    if len(neg_bcs) == 0:
        print(f"WARNING: No negative controls identified. Skipping {res_dir.name}...")
        continue
    
    del_cls.estimate_hyperparameters(negative_barcodes=neg_bcs)
    
    for gene in hrp_genes: 
        print(f"Processing for {gene[0:4]}")
        del_cls.run_mcmc(target_gene=gene)
    summary = del_cls.summarise_mcmc_outputs()
    summary["expt_name"] = res_dir.name
    # Join in sample_type
    summary = summary.merge(exp_meta, on="barcode")
    dfs.append(summary)
        
if len(dfs) == 0:
    print("No valid experiments identified")
else:
    deletions_df = pd.concat(dfs, ignore_index=True)
    # Transform into expected output
    field_df = deletions_df.rename(columns={"hrp2_del_prediction": "hrp2", "hrp3_del_prediction": "hrp3"}).copy(deep=True)
    field_df = field_df[field_df["sample_type"] == "field"]
    field_df.drop(columns=["sample_type"],inplace=True)

Processing 2025-06-05_SLMW010_ATSB_BatchA
Processing for hrp2
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing for hrp3
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing 2025-10-27_SLMW014_ATSB_BatchA_Repeat
Processing for hrp2
Initialising...
Iterating... 50000


/home/dan/git/resources/notebooks/gene_deletions/../functions/gene_deletions.py:263: RuntimeWarning:

invalid value encountered in divide



Done.
Final acceptance rate: 2.000040000800016e-05
Processing for hrp3
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing 2025-12-01_SLMW016_repeats
Processing for hrp2
Initialising...
Iterating... 50000


/home/dan/git/resources/notebooks/gene_deletions/../functions/gene_deletions.py:263: RuntimeWarning:

invalid value encountered in divide



Done.
Final acceptance rate: 2.000040000800016e-05
Processing for hrp3
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing 2024-07-30_MALARIA-CDC-EXPT4-MC
Processing for hrp2
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing for hrp3
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing 2024-07-26_MALARIA-CDC-EXPT3-MC-R2
Processing for hrp2
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05
Processing for hrp3
Initialising...
Iterating... 50000
Done.
Final acceptance rate: 2.000040000800016e-05


In [13]:
long_df = field_df.melt(
    id_vars='sample_id',
    value_vars=['hrp2', 'hrp3'],
    var_name='gene',
    value_name='is_deleted'
)

In [14]:
final_del_df = (
    long_df
    .groupby(['sample_id', 'gene'])['is_deleted']
    .agg(
        n_deleted='sum',   
        n_replicates='count',
        is_deleted='max'
    )
    .reset_index()
)

In [15]:
if save_results:
    final_del_df.to_csv(output_dir / "gene_deletions_prediction.csv", index=False)

# Plots

In [16]:
generate_deletion_prevalence_barchart(final_del_df, master_df)